In [32]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
from sklearn.model_selection import train_test_split

# Clinical Data
df = pd.read_csv("./X_train/clinical_train.csv")
df_eval = pd.read_csv("./X_test/clinical_test.csv")

# Molecular Data
maf_df = pd.read_csv("./X_train/molecular_train.csv")
maf_eval = pd.read_csv("./X_test/molecular_test.csv")

target_df = pd.read_csv("./X_target/target_train.csv")
#target_df_test = pd.read_csv("./X_target/target_test.csv")

# Preview the data
df.head()

,ID,CENTER,BM_BLAST,WBC,ANC,MONOCYTES,HB,PLT,CYTOGENETICS
0,P132697,MSK,14.0,2.8,0.2,0.7,7.6,119.0,"46,xy,del(20)(q12)[2]/46,xy[18]"
1,P132698,MSK,1.0,7.4,2.4,0.1,11.6,42.0,"46,xx"
2,P116889,MSK,15.0,3.7,2.1,0.1,14.2,81.0,"46,xy,t(3;3)(q25;q27)[8]/46,xy[12]"
3,P132699,MSK,1.0,3.9,1.9,0.1,8.9,77.0,"46,xy,del(3)(q26q27)[15]/46,xy[5]"
4,P132700,MSK,6.0,128.0,9.7,0.9,11.1,195.0,"46,xx,t(3;9)(p13;q22)[10]/46,xx[10]"


In [33]:
# handling cytogenics
import pandas as pd
import re

# create a function to parse the ISCN strings
def parse_cytogenetics(val):
    if pd.isna(val):
        return 'Unknown', 'None', 0.0
    
    val = str(val).strip()
    
    # FIX: check if completely normal, allowing for optional bracketed counts like [20]
    if re.match(r'^46,x[xy](\[\d+\])?$', val, re.IGNORECASE):
        return 'Normal', 'None', 0.0
        
    # check for mosaic structures containing bracketed cell counts: e.g., ...[10]/...[10]
    mosaic_match = re.search(r'\[(\d+)\].*/.*\[(\d+)\]', val)
    if mosaic_match:
        abnormal_cells = int(mosaic_match.group(1))
        normal_cells = int(mosaic_match.group(2))
        total_cells = abnormal_cells + normal_cells
        abnormal_pct = (abnormal_cells / total_cells) * 100
        
        # FIX: Isolate the specific mutation string correctly without string splitting bugs
        first_clone = val.split('/')[0]
        mutation = re.sub(r'^4[5678],x[xy],?', '', first_clone, flags=re.IGNORECASE).split('[')[0].strip()
        
        return 'Mosaic', mutation, round(abnormal_pct, 1)
        
    # fallback for non-mosaic abnormal strings
    return 'Abnormal (Non-mosaic)', 'Complex/Other', 100.0

# apply the parsing function to your DataFrame
parsed_results = df['CYTOGENETICS'].apply(parse_cytogenetics)

# split the results into new descriptive columns
df['cytogenetic_status'] = [res[0] for res in parsed_results]
df['extracted_mutation'] = [res[1] for res in parsed_results]
df['abnormal_cell_pct']  = [res[2] for res in parsed_results]

# view the updated DataFrame
df.head(10)


,ID,CENTER,BM_BLAST,WBC,ANC,MONOCYTES,HB,PLT,CYTOGENETICS,cytogenetic_status,extracted_mutation,abnormal_cell_pct
0,P132697,MSK,14.0,2.8,0.2,0.70,7.6,119.0,"46,xy,del(20)(q12)[2]/46,xy[18]",Mosaic,del(20)(q12),10.0
1,P132698,MSK,1.0,7.4,2.4,0.10,11.6,42.0,"46,xx",Normal,None,0.0
2,P116889,MSK,15.0,3.7,2.1,0.10,14.2,81.0,"46,xy,t(3;3)(q25;q27)[8]/46,xy[12]",Mosaic,t(3;3)(q25;q27),40.0
3,P132699,MSK,1.0,3.9,1.9,0.10,8.9,77.0,"46,xy,del(3)(q26q27)[15]/46,xy[5]",Mosaic,del(3)(q26q27),75.0
4,P132700,MSK,6.0,128.0,9.7,0.90,11.1,195.0,"46,xx,t(3;9)(p13;q22)[10]/46,xx[10]",Mosaic,t(3;9)(p13;q22),50.0
5,P132701,MSK,3.0,15.8,12.6,0.88,8.0,492.0,"46,xx[20]",Normal,None,0.0
6,P132702,MSK,12.0,6.2,3.0,0.80,9.6,61.0,"46,xy[20]",Normal,None,0.0
7,P132703,MSK,15.0,4.8,1.5,0.70,9.4,142.0,"46,xy[20]",Normal,None,0.0
8,P132704,MSK,14.0,0.8,0.2,0.00,8.8,23.0,"45,xx,del(5)(q13q33),inc[2]/46,xx[2]",Mosaic,"del(5)(q13q33),inc",50.0
9,P132705,MSK,14.0,1.4,0.2,0.00,7.4,183.0,"46,xx[20]",Normal,None,0.0


In [34]:
#tree based so one col
status_map = {'Normal': 0, 'Mosaic': 1, 'Abnormal (Non-mosaic)': 2, 'Unknown': 3}
df['cytogenetic_status'] = df['cytogenetic_status'].map(status_map)
df.head(10)

,ID,CENTER,BM_BLAST,WBC,ANC,MONOCYTES,HB,PLT,CYTOGENETICS,cytogenetic_status,extracted_mutation,abnormal_cell_pct
0,P132697,MSK,14.0,2.8,0.2,0.70,7.6,119.0,"46,xy,del(20)(q12)[2]/46,xy[18]",1,del(20)(q12),10.0
1,P132698,MSK,1.0,7.4,2.4,0.10,11.6,42.0,"46,xx",0,None,0.0
2,P116889,MSK,15.0,3.7,2.1,0.10,14.2,81.0,"46,xy,t(3;3)(q25;q27)[8]/46,xy[12]",1,t(3;3)(q25;q27),40.0
3,P132699,MSK,1.0,3.9,1.9,0.10,8.9,77.0,"46,xy,del(3)(q26q27)[15]/46,xy[5]",1,del(3)(q26q27),75.0
4,P132700,MSK,6.0,128.0,9.7,0.90,11.1,195.0,"46,xx,t(3;9)(p13;q22)[10]/46,xx[10]",1,t(3;9)(p13;q22),50.0
5,P132701,MSK,3.0,15.8,12.6,0.88,8.0,492.0,"46,xx[20]",0,None,0.0
6,P132702,MSK,12.0,6.2,3.0,0.80,9.6,61.0,"46,xy[20]",0,None,0.0
7,P132703,MSK,15.0,4.8,1.5,0.70,9.4,142.0,"46,xy[20]",0,None,0.0
8,P132704,MSK,14.0,0.8,0.2,0.00,8.8,23.0,"45,xx,del(5)(q13q33),inc[2]/46,xx[2]",1,"del(5)(q13q33),inc",50.0
9,P132705,MSK,14.0,1.4,0.2,0.00,7.4,183.0,"46,xx[20]",0,None,0.0


In [35]:
# display unique values with counts
print(df['extracted_mutation'].value_counts(dropna=False))


extracted_mutation
None                                                                                            2076
Complex/Other                                                                                    429
+8                                                                                               103
45,x,-y                                                                                           65
del(5)(q13q33)                                                                                    46
                                                                                                ... 
del(5)(q13q33),-7,add(17)(q25),del(20)(q11.2)                                                      1
del(5)(q12q35),del(6)(q12q25),del(7)(q22),add(9)(p22),add(11)(q23),add(12)(p12),-15,-16,+mar       1
-7,add(12)(p13),-17,add(17)(q21),-20,+2mar                                                         1
+10                                                                     

since there are 417 unique mutation strings, if you try to use a Linear Model or a standard Neural Network, you have to one-hot encode them. This adds 417 new columns to the df. Algorithms like XGBoost and LightGBM have built-in missing value splits.

In [36]:
status_counts = df['cytogenetic_status'].value_counts(dropna=False)
status_pcts = df['cytogenetic_status'].value_counts(normalize=True, dropna=False) * 100

# 2. Print out the exact summary breakdown
print("--- Cytogenetic Status Breakdown ---")
for category in status_counts.index:
    # Map the numerical key back to its name for readability in the print statement
    reverse_map = {0: 'Normal (0.0)', 1: 'Mosaic (1.0)', 2: 'Abnormal (2.0)', 3: 'Unknown (3.0)'}
    cat_name = reverse_map.get(category, "Missing/NaN")
    print(f"{cat_name:<20} | Count: {status_counts[category]:<5} | Percentage: {status_pcts[category]:.2f}%")

# 3. Explicitly isolate the 'Unknown' status metrics for your baseline tracking
# FIX: Compare directly to the encoded integer value 3 instead of the string 'unknown'
unknown_mask = df['cytogenetic_status'] == 3
total_unknown = unknown_mask.sum()
pct_unknown = unknown_mask.mean() * 100

print("\n--- Unknown Status Metrics ---")
print(f"Total Unknown patients: {total_unknown}")
print(f"Percentage of dataset:  {pct_unknown:.2f}%")

# Preview the final dataframe status
df.head(10)


--- Cytogenetic Status Breakdown ---
Normal (0.0)         | Count: 1689  | Percentage: 50.83%
Mosaic (1.0)         | Count: 818   | Percentage: 24.62%
Abnormal (2.0)       | Count: 429   | Percentage: 12.91%
Unknown (3.0)        | Count: 387   | Percentage: 11.65%

--- Unknown Status Metrics ---
Total Unknown patients: 387
Percentage of dataset:  11.65%


,ID,CENTER,BM_BLAST,WBC,ANC,MONOCYTES,HB,PLT,CYTOGENETICS,cytogenetic_status,extracted_mutation,abnormal_cell_pct
0,P132697,MSK,14.0,2.8,0.2,0.70,7.6,119.0,"46,xy,del(20)(q12)[2]/46,xy[18]",1,del(20)(q12),10.0
1,P132698,MSK,1.0,7.4,2.4,0.10,11.6,42.0,"46,xx",0,None,0.0
2,P116889,MSK,15.0,3.7,2.1,0.10,14.2,81.0,"46,xy,t(3;3)(q25;q27)[8]/46,xy[12]",1,t(3;3)(q25;q27),40.0
3,P132699,MSK,1.0,3.9,1.9,0.10,8.9,77.0,"46,xy,del(3)(q26q27)[15]/46,xy[5]",1,del(3)(q26q27),75.0
4,P132700,MSK,6.0,128.0,9.7,0.90,11.1,195.0,"46,xx,t(3;9)(p13;q22)[10]/46,xx[10]",1,t(3;9)(p13;q22),50.0
5,P132701,MSK,3.0,15.8,12.6,0.88,8.0,492.0,"46,xx[20]",0,None,0.0
6,P132702,MSK,12.0,6.2,3.0,0.80,9.6,61.0,"46,xy[20]",0,None,0.0
7,P132703,MSK,15.0,4.8,1.5,0.70,9.4,142.0,"46,xy[20]",0,None,0.0
8,P132704,MSK,14.0,0.8,0.2,0.00,8.8,23.0,"45,xx,del(5)(q13q33),inc[2]/46,xx[2]",1,"del(5)(q13q33),inc",50.0
9,P132705,MSK,14.0,1.4,0.2,0.00,7.4,183.0,"46,xx[20]",0,None,0.0
